# Chapter 7 — Unified Data Generation Notebook

**Input :** `thesis_data_20.json` (generated by `extract_thesis_data.py`)  
**Outputs:** 29 LaTeX table files, 5 TikZ chart files, 5 matplotlib figures  

Run all cells top-to-bottom to regenerate every Chapter 7 artefact from the authoritative JSON.

## §0  Configuration

In [ ]:
import sys, json, os, math, statistics, warnings
from pathlib import Path

# ── Configurable paths ──────────────────────────────────────────────────────
SCRIPTS_DIR     = Path(__file__).parent if '__file__' in dir() else Path('.')
DATA_JSON       = SCRIPTS_DIR / '../thesis_data_20.json'
THESIS_ROOT     = SCRIPTS_DIR / '../../Grabowski124296mag'
THESIS_TABLES   = THESIS_ROOT / 'tables/ch07'
THESIS_DIAGRAMS = THESIS_ROOT / 'diagrams/ch07'
THESIS_FIGURES  = THESIS_ROOT / 'figures/ch07'

for p in [DATA_JSON, THESIS_TABLES, THESIS_DIAGRAMS, THESIS_FIGURES]:
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p.resolve()}")

print("✓ All paths resolved:")
for label, p in [("DATA_JSON", DATA_JSON), ("THESIS_TABLES", THESIS_TABLES),
                  ("THESIS_DIAGRAMS", THESIS_DIAGRAMS), ("THESIS_FIGURES", THESIS_FIGURES)]:
    print(f"  {label:<16} = {p.resolve()}")


## §1  Load & Validate JSON

In [ ]:
with open(DATA_JSON, 'r', encoding='utf-8') as f:
    DATA = json.load(f)

REQUIRED_KEYS = [
    'overview', 'static_tools', 'dynamic_zap', 'dynamic_diagnostics',
    'performance', 'ai_compliance', 'performance_tools', 'ai_tools',
    'dynamic_tools', 'model_summary',
]
missing = [k for k in REQUIRED_KEYS if k not in DATA]
assert not missing, f"Missing keys: {missing}"
print(f"✓ All {len(REQUIRED_KEYS)} required top-level keys present")

assert len(DATA['model_summary']) == 10, "Expected 10 models"
print("✓ 10 models found")

ab_pm = DATA['performance_tools']['ab']['per_model']
total_deployed = sum(int(v['runs']) // 2 for v in ab_pm.values() if v is not None)
assert total_deployed == 117, f"Expected 117 deployed apps, got {total_deployed}"
print(f"✓ Total deployed apps = {total_deployed} (expected 117)")

print()
print("✓ JSON validation PASSED")


## §2  Import generation functions from `generate_thesis_tables.py`

In [ ]:
import importlib.util

def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

gtt = _load_module('generate_thesis_tables', SCRIPTS_DIR / 'generate_thesis_tables.py')

MODEL_ORDER   = gtt.MODEL_ORDER
SHORT_NAMES   = gtt.SHORT_NAMES
MODEL_PARAMS  = gtt.MODEL_PARAMS

_get_deploy_pcts    = gtt._get_deploy_pcts
_get_compl_pcts     = gtt._get_compl_pcts
_get_quality_scores = gtt._get_quality_scores
_latex_int          = gtt._latex_int
_latex_float        = gtt._latex_float
_sn                 = gtt._sn

_gen_service_completion_table = gtt._gen_service_completion_table
_gen_code_composition_table   = gtt._gen_code_composition_table
_gen_severity_table           = gtt._gen_severity_table
_gen_zap_table                = gtt._gen_zap_table
_gen_curl_endpoint_table      = gtt._gen_curl_endpoint_table
_gen_heatmap_table            = gtt._gen_heatmap_table
_gen_topsis_table             = gtt._gen_topsis_table
_gen_wsm_table                = gtt._gen_wsm_table
_gen_correlation_table        = gtt._gen_correlation_table
_gen_findings_tool_table      = gtt._gen_findings_tool_table
_gen_perf_tool_table          = gtt._gen_perf_tool_table
_gen_ai_tool_table            = gtt._gen_ai_tool_table
_gen_diagnostic_tool_table    = gtt._gen_diagnostic_tool_table
_gen_ai_compliance_summary    = gtt._gen_ai_compliance_summary
_gen_reproducibility_tables   = gtt._gen_reproducibility_tables

_obs_severity         = gtt._obs_severity
_obs_code_composition = gtt._obs_code_composition
_obs_findings_tool    = gtt._obs_findings_tool
_obs_zap              = gtt._obs_zap
_obs_perf             = gtt._obs_perf
_obs_ai_tool          = gtt._obs_ai_tool
_obs_ai_compliance    = gtt._obs_ai_compliance
_obs_topsis           = gtt._obs_topsis
_obs_wsm              = gtt._obs_wsm
_obs_correlation      = gtt._obs_correlation

print("✓ All generation functions imported from generate_thesis_tables.py")


## §3  File-writing helpers

In [ ]:
WRITTEN_FILES = []

def write_table(filename, content):
    path = THESIS_TABLES / filename
    path.write_text(content, encoding='utf-8')
    WRITTEN_FILES.append(('table', path))
    print(f"  ✓ {filename}  ({len(content):,} chars)")

def write_diagram(filename, content):
    path = THESIS_DIAGRAMS / filename
    path.write_text(content, encoding='utf-8')
    WRITTEN_FILES.append(('diagram', path))
    print(f"  ✓ {filename}  ({len(content):,} chars)")

def write_figure(fig, filename):
    path = THESIS_FIGURES / filename
    fig.savefig(path, dpi=150, bbox_inches='tight', format='jpeg')
    import matplotlib.pyplot as plt
    plt.close(fig)
    WRITTEN_FILES.append(('figure', path))
    print(f"  ✓ {filename}")


## §4  Overview & service tables

In [ ]:
print("Generating overview tables...")
write_table('service_completion.tex',
    _gen_service_completion_table(DATA))
write_table('code_composition.tex',
    _gen_code_composition_table(DATA, _obs_code_composition(DATA)))
write_table('severity_by_model.tex',
    _gen_severity_table(DATA, observation=_obs_severity(DATA)))


## §5  Analysis tables (heatmap, correlations, TOPSIS, WSM)

In [ ]:
print("Generating analysis tables...")
write_table('heatmap.tex',
    _gen_heatmap_table(DATA))
write_table('correlations.tex',
    _gen_correlation_table(DATA, observation=_obs_correlation(DATA)))
write_table('topsis.tex',
    _gen_topsis_table(DATA, observation=_obs_topsis(DATA)))
write_table('wsm.tex',
    _gen_wsm_table(DATA, observation=_obs_wsm(DATA)))


## §6  Model parameters table

In [ ]:
print("Generating model_params.tex...")
write_table('model_params.tex', _gen_reproducibility_tables(DATA))


## §7  Static tool tables

In [ ]:
print("Generating static tool tables...")

STATIC_TOOLS = {
    'bandit':         'tool_bandit.tex',
    'semgrep':        'tool_semgrep.tex',
    'pylint':         'tool_pylint.tex',
    'ruff':           'tool_ruff.tex',
    'mypy':           'tool_mypy.tex',
    'vulture':        'tool_vulture.tex',
    'radon':          'tool_radon.tex',
    'safety':         'tool_safety.tex',
    'pip-audit':      'tool_pipaudit.tex',
    'detect-secrets': 'tool_detectsecrets.tex',
    'eslint':         'tool_eslint.tex',
    'npm-audit':      'tool_npmaudit.tex',
}

STATIC_NOTES = {
    'detect-secrets': (
        'Zero findings across all models. LLM-generated code contained no '
        'detectable hardcoded secrets under the default ruleset.'
    ),
}
STATIC_INFO = {
    'bandit':         ('Bandit: Python Security Linter Results by Model',                    'tab:tool_bandit'),
    'semgrep':        ('Semgrep: Pattern-Based Security Analysis Results by Model',          'tab:tool_semgrep'),
    'pylint':         ('Pylint: Python Code Quality Results by Model',                       'tab:tool_pylint'),
    'ruff':           ('Ruff: Python Linting Results by Model',                              'tab:tool_ruff'),
    'mypy':           ('Mypy: Python Type Checking Results by Model',                        'tab:tool_mypy'),
    'vulture':        ('Vulture: Dead Code Detection Results by Model',                      'tab:tool_vulture'),
    'radon':          ('Radon: Python Complexity Analysis Results by Model',                 'tab:tool_radon'),
    'safety':         ('Safety: Python Dependency Vulnerability Results by Model',           'tab:tool_safety'),
    'pip-audit':      ('Pip-audit: Python Package Audit Results by Model',                   'tab:tool_pipaudit'),
    'detect-secrets': ('Detect-secrets: Secret Detection Results by Model',                  'tab:tool_detectsecrets'),
    'eslint':         ('ESLint: JavaScript Linting Results by Model',                        'tab:tool_eslint'),
    'npm-audit':      ('npm-audit: JavaScript Dependency Vulnerability Results by Model',    'tab:tool_npmaudit'),
}
loc_data = DATA['model_summary']
for tool, fname in STATIC_TOOLS.items():
    cap, lbl = STATIC_INFO[tool]
    tdata = DATA['static_tools'].get(tool, {})
    obs   = _obs_findings_tool(tool, tdata, loc_data)
    note  = STATIC_NOTES.get(tool, '')
    write_table(fname, _gen_findings_tool_table(tool, tdata, loc_data, cap, lbl, note=note, observation=obs))

print(f"  -> {len(STATIC_TOOLS)} static tool tables written")


## §8  Dynamic tool tables (ZAP, curl)

In [ ]:
print("Generating dynamic tool tables...")

write_table('tool_zap.tex', _gen_zap_table(DATA, observation=_obs_zap(DATA)))

curl_data = DATA['dynamic_tools'].get('curl', {})
curl_obs  = _obs_findings_tool('curl', curl_data, DATA['model_summary'])
write_table('tool_curl.tex',
    _gen_findings_tool_table('curl', curl_data, DATA['model_summary'],
        'Curl: HTTP Probe Results by Model', 'tab:tool_curl', observation=curl_obs))


## §9  Performance tool tables (ab, locust, artillery, aiohttp)

In [ ]:
print("Generating performance tool tables...")

PERF_TOOLS = {
    'ab':       'tool_ab.tex',
    'locust':   'tool_locust.tex',
    'artillery':'tool_artillery.tex',
    'aiohttp':  'tool_aiohttp.tex',
}
PERF_INFO = {
    'ab':        ('Apache Bench: Load Testing Results by Model', 'tab:tool_ab'),
    'locust':    ('Locust: Load Testing Results by Model',       'tab:tool_locust'),
    'artillery': ('Artillery: Load Testing Results by Model',    'tab:tool_artillery'),
    'aiohttp':   ('aiohttp: Async HTTP Testing Results by Model','tab:tool_aiohttp'),
}
for tool, fname in PERF_TOOLS.items():
    cap, lbl = PERF_INFO[tool]
    tdata = DATA['performance_tools'].get(tool, {})
    write_table(fname, _gen_perf_tool_table(tool, tdata, cap, lbl, observation=_obs_perf(tool, tdata)))

# Combined tables
def _combined(tools):
    parts = []
    for t in tools:
        td = DATA['performance_tools'].get(t, {})
        cap, lbl = PERF_INFO[t]
        parts.append(_gen_perf_tool_table(t, td, cap, lbl, observation=_obs_perf(t, td)))
    return '\n\n'.join(parts)

write_table('tool_perf_ab_locust.tex',          _combined(['ab', 'locust']))
write_table('tool_perf_artillery_aiohttp.tex',  _combined(['artillery', 'aiohttp']))
print(f"  → {len(PERF_TOOLS)} individual + 2 combined perf tables written")


## §10  AI tool tables

In [ ]:
print("Generating AI tool tables...")

ai = DATA.get('ai_tools', {})
rs_data = ai.get('requirements-scanner', {})
cq_data = ai.get('code-quality-analyzer', {})

rs_tex = _gen_ai_tool_table('requirements-scanner', rs_data,
    'Requirements Scanner: Compliance Check Results by Model', 'tab:tool_reqscanner',
    observation=_obs_ai_tool('requirements-scanner', rs_data))
cq_tex = _gen_ai_tool_table('code-quality-analyzer', cq_data,
    'Code Quality Analyzer: AI Review Results by Model', 'tab:tool_codequalanalyzer',
    observation=_obs_ai_tool('code-quality-analyzer', cq_data))
write_table('tool_reqscanner_codeanalyzer.tex', rs_tex + '\n\n' + cq_tex)

write_table('ai_compliance_summary.tex',
    _gen_ai_compliance_summary(DATA, observation=_obs_ai_compliance(DATA)))
print("  → AI tool tables written")


## §11  TikZ chart generation

### §11a  chart_static_overview.tex

In [ ]:
print("Generating chart_static_overview.tex...")

# Short TikZ labels for x-axis
TIKZ_NAME = {
    'anthropic_claude-4.5-sonnet-20250929':            'Claude~4.5',
    'z-ai_glm-4.7-20251222':                            'GLM-4.7',
    'deepseek_deepseek-r1-0528':                        'DeepSeek~R1',
    'openai_gpt-5.2-codex-20260114':                   'GPT-5.2',
    'qwen_qwen3-coder-plus':                            'Qwen3',
    'google_gemini-3-pro-preview-20251117':             'Gemini~Pro',
    'google_gemini-3-flash-preview-20251217':           'Gemini~Flash',
    'mistralai_mistral-small-3.1-24b-instruct-2503':   'Mistral',
    'meta-llama_llama-3.1-405b-instruct':               'Llama~405B',
    'openai_gpt-4o-mini':                               'GPT-4o~Mini',
}

# Aggregate H/M/L across all static tools (totals, then /20 for avg/app)
static_pm = {s: {'H': 0, 'M': 0, 'L': 0} for s in MODEL_ORDER}
for tdata in DATA['static_tools'].values():
    for slug, pm in tdata.get('per_model', {}).items():
        if slug in static_pm:
            static_pm[slug]['H'] += pm.get('sv_high',   pm.get('high',   0))
            static_pm[slug]['M'] += pm.get('sv_medium',  pm.get('medium', 0))
            static_pm[slug]['L'] += pm.get('sv_low',     pm.get('low',    0))

# Sort by total desc
sorted_slugs = sorted(MODEL_ORDER,
    key=lambda s: static_pm[s]['H'] + static_pm[s]['M'] + static_pm[s]['L'], reverse=True)

def _coord(slug, key):
    return f"({TIKZ_NAME[slug]}, {static_pm[slug][key]/20:.1f})"

low_c  = '\n    '.join(_coord(s, 'L') for s in sorted_slugs)
med_c  = '\n    '.join(_coord(s, 'M') for s in sorted_slugs)
high_c = '\n    '.join(
    f"({TIKZ_NAME[s]}, {static_pm[s]['H']/20:.1f}) [{round((static_pm[s]['H']+static_pm[s]['M']+static_pm[s]['L'])/20)}]"
    for s in sorted_slugs)

x_coords = ', '.join(TIKZ_NAME[s] for s in sorted_slugs)
ymax = math.ceil((max(static_pm[s]['H']+static_pm[s]['M']+static_pm[s]['L'] for s in MODEL_ORDER)/20)/50)*50+30

lines = [
    r'% Define chart colors',
    r'\definecolor{sevhigh}{HTML}{C0392B}',
    r'\definecolor{sevmed}{HTML}{E67E22}',
    r'\definecolor{sevlow}{HTML}{2980B9}',
    '',
    r'\begin{figure}[htbp]',
    r'\centering',
    r'\begin{tikzpicture}',
    r'\begin{axis}[',
    r'    ybar stacked,',
    r'    width=\textwidth,',
    r'    height=10cm,',
    r'    bar width=14pt,',
    r'    ylabel={Average findings per application},',
    r'    ylabel style={font=\small},',
    f'    symbolic x coords={{{x_coords}}},',
    r'    xtick=data,',
    r'    x tick label style={rotate=40, anchor=east, font=\footnotesize},',
    r'    y tick label style={font=\small},',
    f'    ymin=0, ymax={ymax},',
    r'    enlarge x limits=0.07,',
    r'    legend style={at={(0.5,1.02)}, anchor=south, legend columns=3, font=\small, draw=gray!40, fill=white, fill opacity=0.9},',
    r'    grid=major, ymajorgrids=true, xmajorgrids=false,',
    r'    major grid style={line width=.2pt, draw=gray!25},',
    r'    axis line style={draw=gray!60},',
    r'    tick style={draw=gray!60},',
    r'    nodes near coords,',
    r'    nodes near coords style={font=\tiny, color=black!70},',
    r'    every node near coord/.append style={anchor=south, yshift=0pt},',
    r'    point meta=explicit symbolic,',
    r']',
    '',
    r'% Low severity (bottom)',
    r'\addplot+[fill=sevlow, draw=sevlow!80, nodes near coords style={opacity=0}] coordinates {',
    f'    {low_c}',
    r'};',
    '',
    r'% Medium severity (middle)',
    r'\addplot+[fill=sevmed, draw=sevmed!80, nodes near coords style={opacity=0}] coordinates {',
    f'    {med_c}',
    r'};',
    '',
    r'% High severity (top) --- with total labels',
    r'\addplot+[fill=sevhigh, draw=sevhigh!80] coordinates {',
    f'    {high_c}',
    r'};',
    '',
    r'\legend{Low, Medium, High}',
    r'\end{axis}',
    r'\end{tikzpicture}',
    r'\caption{Average static-analysis findings per application, stacked by severity. Models ordered by total findings (descending). High-severity findings (red) represent the most critical vulnerabilities.}',
    r'\label{fig:chart-static-overview}',
    r'\end{figure}',
]
write_diagram('chart_static_overview.tex', '\n'.join(lines))


### §11b  chart_defect_density.tex

In [ ]:
print("Generating chart_defect_density.tex...")

def sv_sum_dkloc(slug):
    total_loc = DATA['model_summary'][slug].get('total_loc', 1)
    sv = 0
    for tdata in DATA['static_tools'].values():
        pm = tdata.get('per_model', {}).get(slug, {})
        sv += pm.get('sv_high', pm.get('high', 0))
        sv += pm.get('sv_medium', pm.get('medium', 0))
        sv += pm.get('sv_low', pm.get('low', 0))
    return sv / total_loc * 1000 if total_loc else 0

def sv_sum_dkfunc(slug):
    ms = DATA['model_summary'][slug]
    func_loc = ms.get('python_loc', 0) + ms.get('js_loc', 0)
    if func_loc == 0:
        return 0
    sv = 0
    for tdata in DATA['static_tools'].values():
        pm = tdata.get('per_model', {}).get(slug, {})
        sv += pm.get('sv_high', pm.get('high', 0))
        sv += pm.get('sv_medium', pm.get('medium', 0))
        sv += pm.get('sv_low', pm.get('low', 0))
    return sv / func_loc * 1000

# Sort by D/kLOC ascending
slugs_d = sorted(MODEL_ORDER, key=sv_sum_dkloc)
names_d = [TIKZ_NAME[s] for s in slugs_d]
x_d = ', '.join(names_d)

kloc_c  = '\n    '.join(f"({TIKZ_NAME[s]}, {sv_sum_dkloc(s):.1f})" for s in slugs_d)
kfunc_c = '\n    '.join(f"({TIKZ_NAME[s]}, {sv_sum_dkfunc(s):.1f})" for s in slugs_d)
ymax_d = math.ceil(max(sv_sum_dkfunc(s) for s in MODEL_ORDER) / 50) * 50 + 20

lines2 = [
    r'% Define chart colors',
    r'\definecolor{colkloc}{HTML}{2980B9}',
    r'\definecolor{colkfunc}{HTML}{C0392B}',
    '',
    r'\begin{figure}[htbp]',
    r'\centering',
    r'\begin{tikzpicture}',
    r'\begin{axis}[',
    r'    ybar,',
    r'    width=\textwidth,',
    r'    height=10cm,',
    r'    bar width=10pt,',
    r'    ylabel={Findings per kLOC},',
    r'    ylabel style={font=\small},',
    f'    symbolic x coords={{{x_d}}},',
    r'    xtick=data,',
    r'    x tick label style={rotate=40, anchor=east, font=\footnotesize},',
    r'    y tick label style={font=\small},',
    f'    ymin=0, ymax={ymax_d},',
    r'    enlarge x limits=0.07,',
    r'    legend style={at={(0.02,0.97)}, anchor=north west, legend columns=1, font=\small, draw=gray!40, fill=white, fill opacity=0.9},',
    r'    grid=major, ymajorgrids=true, xmajorgrids=false,',
    r'    major grid style={line width=.2pt, draw=gray!25},',
    r'    axis line style={draw=gray!60},',
    r'    tick style={draw=gray!60},',
    r'    nodes near coords,',
    r'    nodes near coords style={font=\tiny, rotate=90, anchor=west, color=black!60},',
    r']',
    '',
    r'% D/kLOC (total LOC including scaffolding)',
    r'\addplot+[fill=colkloc, draw=colkloc!80] coordinates {',
    f'    {kloc_c}',
    r'};',
    '',
    r'% D/kFunc (functional LOC only)',
    r'\addplot+[fill=colkfunc, draw=colkfunc!80] coordinates {',
    f'    {kfunc_c}',
    r'};',
    '',
    r'\legend{D/kLOC (total), D/kFunc (functional only)}',
    r'\end{axis}',
    r'\end{tikzpicture}',
    r'\caption{Defect density by normalization method. Models sorted by D/kLOC ascending. D/kFunc uses only Python+JS lines (excluding scaffolding). The gap between bars reflects scaffolding code dilution.}',
    r'\label{fig:chart-defect-density}',
    r'\end{figure}',
]
write_diagram('chart_defect_density.tex', '\n'.join(lines2))


### §11c  chart_dependencies.tex

In [ ]:
print("Generating chart_dependencies.tex...")

def avg_findings_per_run(tool, storage='static_tools'):
    tdata = DATA[storage].get(tool, {})
    res = {}
    for slug in MODEL_ORDER:
        pm = tdata.get('per_model', {}).get(slug, {})
        runs = pm.get('runs', 0)
        finds = pm.get('total_findings', pm.get('findings', 0))
        res[slug] = finds / runs if runs else 0.0
    return res

safety_avg   = avg_findings_per_run('safety')
pipaudit_avg = avg_findings_per_run('pip-audit')
npmaudit_avg = avg_findings_per_run('npm-audit')

slugs_dep = sorted(MODEL_ORDER,
    key=lambda s: safety_avg[s]+pipaudit_avg[s]+npmaudit_avg[s], reverse=True)
x_dep = ', '.join(TIKZ_NAME[s] for s in slugs_dep)
safety_c  = '\n    '.join(f"({TIKZ_NAME[s]}, {safety_avg[s]:.1f})"   for s in slugs_dep)
pip_c     = '\n    '.join(f"({TIKZ_NAME[s]}, {pipaudit_avg[s]:.1f})" for s in slugs_dep)
npm_c     = '\n    '.join(f"({TIKZ_NAME[s]}, {npmaudit_avg[s]:.1f})" for s in slugs_dep)
ymax_dep  = math.ceil(max(safety_avg[s]+pipaudit_avg[s]+npmaudit_avg[s] for s in MODEL_ORDER)) + 3

lines3 = [
    r'% Define chart colors',
    r'\definecolor{colsafety}{HTML}{C0392B}',
    r'\definecolor{colpipaudit}{HTML}{2980B9}',
    r'\definecolor{colnpmaudit}{HTML}{27AE60}',
    '',
    r'\begin{figure}[htbp]',
    r'\centering',
    r'\begin{tikzpicture}',
    r'\begin{axis}[',
    r'    ybar,',
    r'    width=\textwidth,',
    r'    height=9cm,',
    r'    bar width=7pt,',
    r'    ylabel={Average findings per run},',
    r'    ylabel style={font=\small},',
    f'    symbolic x coords={{{x_dep}}},',
    r'    xtick=data,',
    r'    x tick label style={rotate=40, anchor=east, font=\footnotesize},',
    r'    y tick label style={font=\small},',
    f'    ymin=0, ymax={ymax_dep},',
    r'    enlarge x limits=0.07,',
    r'    legend style={at={(0.98,0.97)}, anchor=north east, legend columns=1, font=\small, draw=gray!40, fill=white, fill opacity=0.9},',
    r'    grid=major, ymajorgrids=true, xmajorgrids=false,',
    r'    major grid style={line width=.2pt, draw=gray!25},',
    r'    axis line style={draw=gray!60},',
    r'    tick style={draw=gray!60},',
    r'    nodes near coords,',
    r'    nodes near coords style={font=\tiny, rotate=90, anchor=west, color=black!60},',
    r']',
    '',
    r'% Safety (Python dependencies)',
    r'\addplot+[fill=colsafety, draw=colsafety!80] coordinates {',
    f'    {safety_c}',
    r'};',
    '',
    r'% pip-audit (Python dependencies)',
    r'\addplot+[fill=colpipaudit, draw=colpipaudit!80] coordinates {',
    f'    {pip_c}',
    r'};',
    '',
    r'% npm-audit (JavaScript dependencies)',
    r'\addplot+[fill=colnpmaudit, draw=colnpmaudit!80] coordinates {',
    f'    {npm_c}',
    r'};',
    '',
    r'\legend{Safety (Python deps), pip-audit (Python deps), npm-audit (JS deps)}',
    r'\end{axis}',
    r'\end{tikzpicture}',
    r'\caption{Dependency vulnerability findings per run for safety, pip-audit, and npm-audit. Models sorted by combined total descending. Safety and pip-audit results are similar across models (driven by shared Flask stack); npm-audit findings are consistent for all models.}',
    r'\label{fig:chart-dependencies}',
    r'\end{figure}',
]
write_diagram('chart_dependencies.tex', '\n'.join(lines3))


### §11d  chart_tools_comparison.tex

In [ ]:
print("Generating chart_tools_comparison.tex...")

ruff_avg   = avg_findings_per_run('ruff')
eslint_avg = avg_findings_per_run('eslint')
bandit_avg = avg_findings_per_run('bandit')

slugs_tc = sorted(MODEL_ORDER, key=lambda s: ruff_avg[s], reverse=True)
x_tc = ', '.join(TIKZ_NAME[s] for s in slugs_tc)
ruff_c = '\n    '.join(f"({TIKZ_NAME[s]}, {ruff_avg[s]:.1f})"   for s in slugs_tc)
esl_c  = '\n    '.join(f"({TIKZ_NAME[s]}, {eslint_avg[s]:.1f})" for s in slugs_tc)
ban_c  = '\n    '.join(f"({TIKZ_NAME[s]}, {bandit_avg[s]:.1f})" for s in slugs_tc)
ymax_tc = math.ceil(max(ruff_avg[s]+eslint_avg[s]+bandit_avg[s] for s in MODEL_ORDER)/20)*20+10

lines4 = [
    r'% Define chart colors',
    r'\definecolor{colruff}{HTML}{1ABC9C}',
    r'\definecolor{coleslint}{HTML}{E67E22}',
    r'\definecolor{colbandit}{HTML}{C0392B}',
    '',
    r'\begin{figure}[htbp]',
    r'\centering',
    r'\begin{tikzpicture}',
    r'\begin{axis}[',
    r'    ybar,',
    r'    width=\textwidth,',
    r'    height=10cm,',
    r'    bar width=7pt,',
    r'    ylabel={Average findings per run},',
    r'    ylabel style={font=\small},',
    f'    symbolic x coords={{{x_tc}}},',
    r'    xtick=data,',
    r'    x tick label style={rotate=40, anchor=east, font=\footnotesize},',
    r'    y tick label style={font=\small},',
    f'    ymin=0, ymax={ymax_tc},',
    r'    enlarge x limits=0.07,',
    r'    legend style={at={(0.98,0.97)}, anchor=north east, legend columns=1, font=\small, draw=gray!40, fill=white, fill opacity=0.9},',
    r'    grid=major, ymajorgrids=true, xmajorgrids=false,',
    r'    major grid style={line width=.2pt, draw=gray!25},',
    r'    axis line style={draw=gray!60},',
    r'    tick style={draw=gray!60},',
    r'    nodes near coords,',
    r'    nodes near coords style={font=\tiny, rotate=90, anchor=west, color=black!60},',
    r']',
    '',
    r'% Ruff (Python linter)',
    r'\addplot+[fill=colruff, draw=colruff!80] coordinates {',
    f'    {ruff_c}',
    r'};',
    '',
    r'% ESLint (JavaScript linter)',
    r'\addplot+[fill=coleslint, draw=coleslint!80] coordinates {',
    f'    {esl_c}',
    r'};',
    '',
    r'% Bandit (Python security)',
    r'\addplot+[fill=colbandit, draw=colbandit!80] coordinates {',
    f'    {ban_c}',
    r'};',
    '',
    r'\legend{Ruff (Python lint), ESLint (JS lint), Bandit (Python security)}',
    r'\end{axis}',
    r'\end{tikzpicture}',
    r'\caption{Average findings per run for the three highest-volume code-level tools. Models sorted by Ruff output descending. Ruff identifies Python style/quality issues; ESLint identifies JavaScript issues; Bandit identifies Python security anti-patterns.}',
    r'\label{fig:chart-tools-comparison}',
    r'\end{figure}',
]
write_diagram('chart_tools_comparison.tex', '\n'.join(lines4))


### §11e  chart_compliance.tex

In [ ]:
print("Generating chart_compliance.tex...")

# Per-model backend/frontend/admin compliance from ai_compliance
# Structure: ai_compliance[slug] = {'backend': {'mean': ...}, 'frontend': {...}, 'admin': {...}}
ai_comp = DATA.get('ai_compliance', {})
comp_by_model = {}
for slug, d in ai_comp.items():
    if not isinstance(d, dict) or 'backend' not in d:
        continue
    comp_by_model[slug] = {
        'backend':  round(d['backend']['mean'],  1),
        'frontend': round(d['frontend']['mean'], 1),
        'admin':    round(d['admin']['mean'],    1),
    }
    comp_by_model[slug]['overall'] = round(d.get('overall', {}).get('mean',
        statistics.mean([comp_by_model[slug]['backend'],
                         comp_by_model[slug]['frontend'],
                         comp_by_model[slug]['admin']])), 1)

slugs_comp = sorted(comp_by_model, key=lambda s: comp_by_model[s]['overall'], reverse=True)
x_comp = ', '.join(TIKZ_NAME.get(s, s) for s in slugs_comp)

be_c = '\n    '.join(f"({TIKZ_NAME.get(s,s)}, {comp_by_model[s]['backend']})" for s in slugs_comp)
fe_c = '\n    '.join(f"({TIKZ_NAME.get(s,s)}, {comp_by_model[s]['frontend']})" for s in slugs_comp)
ad_c = '\n    '.join(f"({TIKZ_NAME.get(s,s)}, {comp_by_model[s]['admin']})" for s in slugs_comp)

print("  Per-model overall compliance:")
for s in slugs_comp:
    print(f"    {TIKZ_NAME.get(s,s):<14}: {comp_by_model[s]['overall']:.1f}%")

lines5 = [
    r'% Define chart colors',
    r'\definecolor{colbackend}{HTML}{1ABC9C}',
    r'\definecolor{colfrontend}{HTML}{E74C3C}',
    r'\definecolor{coladmin}{HTML}{8E44AD}',
    '',
    r'\begin{figure}[htbp]',
    r'\centering',
    r'\begin{tikzpicture}',
    r'\begin{axis}[',
    r'    ybar,',
    r'    width=\textwidth,',
    r'    height=10cm,',
    r'    bar width=7pt,',
    r'    ylabel={Compliance (\%)},',
    r'    ylabel style={font=\small},',
    f'    symbolic x coords={{{x_comp}}},',
    r'    xtick=data,',
    r'    x tick label style={rotate=40, anchor=east, font=\footnotesize},',
    r'    y tick label style={font=\small},',
    r'    ymin=40, ymax=105,',
    r'    enlarge x limits=0.07,',
    r'    legend style={at={(0.98,0.03)}, anchor=south east, legend columns=1, font=\small, draw=gray!40, fill=white, fill opacity=0.9},',
    r'    grid=major, ymajorgrids=true, xmajorgrids=false,',
    r'    major grid style={line width=.2pt, draw=gray!25},',
    r'    axis line style={draw=gray!60},',
    r'    tick style={draw=gray!60},',
    r'    nodes near coords,',
    r'    nodes near coords style={font=\tiny, rotate=90, anchor=west, color=black!60},',
    r'    extra x ticks={},',
    r']',
    '',
    r'% Backend compliance',
    r'\addplot+[fill=colbackend, draw=colbackend!80] coordinates {',
    f'    {be_c}',
    r'};',
    '',
    r'% Frontend compliance',
    r'\addplot+[fill=colfrontend, draw=colfrontend!80] coordinates {',
    f'    {fe_c}',
    r'};',
    '',
    r'% Admin compliance',
    r'\addplot+[fill=coladmin, draw=coladmin!80] coordinates {',
    f'    {ad_c}',
    r'};',
    '',
    r'\legend{Backend, Frontend, Admin}',
    r'\end{axis}',
    r'\end{tikzpicture}',
    r'\caption{AI-assessed requirements compliance by component across all models, sorted by overall compliance score. Admin compliance is consistently high ($\geq$80\%) across all models; frontend compliance shows the widest spread.}',
    r'\label{fig:chart-compliance}',
    r'\end{figure}',
]
write_diagram('chart_compliance.tex', '\n'.join(lines5))
print(f"  --> All 5 TikZ charts generated")


## §12  Matplotlib figure generation

In [ ]:
import warnings
warnings.filterwarnings('ignore')

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import numpy as np
    print(f"✓ matplotlib {matplotlib.__version__}, numpy {np.__version__}")
except ImportError as e:
    print(f"✗ Not available: {e}")
    print("  Install: pip install matplotlib 'numpy<2' --break-system-packages")
    raise


### §12a  scorecard.jpg  (TOPSIS-based model ranking)

In [ ]:
# Recompute TOPSIS scores (must match topsis.tex exactly)
deploy_pcts  = _get_deploy_pcts(DATA)
compl_pcts   = _get_compl_pcts(DATA)
quality_sc   = _get_quality_scores(DATA)

WEIGHTS    = [0.30, 0.10, 0.10, 0.20, 0.15, 0.15]
IS_BENEFIT = [True, True, True, True, False, False]

raw = {s: [
    deploy_pcts[s],
    compl_pcts[s],
    quality_sc[s],
    int(DATA['model_summary'][s].get('total_loc', 0) / 20),
    sv_sum_dkloc(s),
    MODEL_PARAMS[s]['out_price'],
] for s in MODEL_ORDER}

norms = [math.sqrt(sum(raw[s][j]**2 for s in MODEL_ORDER)) or 1 for j in range(6)]
wgt = {s: [WEIGHTS[j]*raw[s][j]/norms[j] for j in range(6)] for s in MODEL_ORDER}

ideal = [max(wgt[s][j] for s in MODEL_ORDER) if IS_BENEFIT[j]
         else min(wgt[s][j] for s in MODEL_ORDER) for j in range(6)]
anti  = [min(wgt[s][j] for s in MODEL_ORDER) if IS_BENEFIT[j]
         else max(wgt[s][j] for s in MODEL_ORDER) for j in range(6)]

topsis = {}
for s in MODEL_ORDER:
    dp = math.sqrt(sum((wgt[s][j]-ideal[j])**2 for j in range(6)))
    dm = math.sqrt(sum((wgt[s][j]-anti[j])**2  for j in range(6)))
    topsis[s] = dm / (dp+dm) if (dp+dm) else 0

ranked = sorted(MODEL_ORDER, key=lambda s: topsis[s], reverse=True)
scores = [topsis[s] for s in ranked]
names  = [SHORT_NAMES[s] for s in ranked]
colors = ['#2ecc71' if sc>=0.6 else '#e67e22' if sc>=0.4 else '#e74c3c' for sc in scores]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(names[::-1], scores[::-1], color=colors[::-1])
ax.set_xlabel('TOPSIS Score', fontsize=12)
ax.set_title('Model Ranking — TOPSIS Multi-criteria Score', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.05)
ax.axvline(0.6, color='green',  linestyle='--', alpha=0.5, label='Tier 1 (>=0.6)')
ax.axvline(0.4, color='orange', linestyle='--', alpha=0.5, label='Tier 2 (>=0.4)')
for bar, sc in zip(bars[::-1], scores[::-1]):
    ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
            f'{sc:.4f}', va='center', fontsize=10)
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
write_figure(fig, 'scorecard.jpg')
print("  Top 3:", ', '.join(f"{SHORT_NAMES[s]}={topsis[s]:.4f}" for s in ranked[:3]))


### §12b  severity_distribution.jpg

In [ ]:
sev_data = {}
for slug in MODEL_ORDER:
    h = m = l = 0
    for tdata in DATA['static_tools'].values():
        pm = tdata.get('per_model', {}).get(slug, {})
        h += pm.get('sv_high',   pm.get('high',   0))
        m += pm.get('sv_medium',  pm.get('medium', 0))
        l += pm.get('sv_low',     pm.get('low',    0))
    sev_data[slug] = (h, m, l)

by_total = sorted(MODEL_ORDER, key=lambda s: sum(sev_data[s]), reverse=True)
names_s  = [SHORT_NAMES[s] for s in by_total]
highs    = [sev_data[s][0] for s in by_total]
meds     = [sev_data[s][1] for s in by_total]
lows     = [sev_data[s][2] for s in by_total]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(names_s))
w = 0.6
ax.bar(x, lows,  w, label='Low',    color='#3498db')
ax.bar(x, meds,  w, label='Medium', color='#e67e22', bottom=lows)
ax.bar(x, highs, w, label='High',   color='#c0392b',
       bottom=[lo+me for lo, me in zip(lows, meds)])
ax.set_xticks(x); ax.set_xticklabels(names_s, rotation=35, ha='right')
ax.set_ylabel('Total Findings')
ax.set_title('Static Analysis Findings by Severity and Model', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
write_figure(fig, 'severity_distribution.jpg')


### §12c  static_heatmap.jpg

In [ ]:
HEAT_TOOLS = [
    'bandit','semgrep','pylint','ruff','mypy',
    'vulture','radon','safety','pip-audit','detect-secrets','eslint','npm-audit'
]
heat = np.zeros((len(HEAT_TOOLS), len(MODEL_ORDER)))
for ti, tool in enumerate(HEAT_TOOLS):
    tdata = DATA['static_tools'].get(tool, {})
    for mi, slug in enumerate(MODEL_ORDER):
        pm = tdata.get('per_model', {}).get(slug, {})
        runs  = pm.get('runs', 0)
        finds = pm.get('total_findings', pm.get('findings', 0))
        heat[ti, mi] = finds/runs if runs else 0

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(heat, aspect='auto', cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Avg findings / run')
ax.set_xticks(range(len(MODEL_ORDER)))
ax.set_xticklabels([SHORT_NAMES[s] for s in MODEL_ORDER], rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(HEAT_TOOLS)))
ax.set_yticklabels(HEAT_TOOLS)
ax.set_title('Static Analysis Tools x Models Heatmap (avg findings/run)', fontsize=13, fontweight='bold')
for ti in range(len(HEAT_TOOLS)):
    for mi in range(len(MODEL_ORDER)):
        v = heat[ti, mi]
        ax.text(mi, ti, f'{v:.1f}', ha='center', va='center',
                fontsize=7, color='black' if v < heat.max()*0.6 else 'white')
plt.tight_layout()
write_figure(fig, 'static_heatmap.jpg')


### §12d  volume_vs_defects.jpg

In [ ]:
deploy_pcts_fig = _get_deploy_pcts(DATA)
locs    = [DATA['model_summary'][s].get('total_loc', 0)/20 for s in MODEL_ORDER]
dklocs  = [sv_sum_dkloc(s) for s in MODEL_ORDER]
names_v = [SHORT_NAMES[s] for s in MODEL_ORDER]
deploys = [deploy_pcts_fig[s] for s in MODEL_ORDER]

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(locs, dklocs, s=[d*3+30 for d in deploys],
                c=deploys, cmap='RdYlGn', vmin=0, vmax=100,
                edgecolors='black', linewidth=0.5, alpha=0.85)
plt.colorbar(sc, ax=ax, label='Deploy % (colour)', shrink=0.7)
ax.set_xlabel('Average LOC per App', fontsize=11)
ax.set_ylabel('D/kLOC (sv_sum-based)', fontsize=11)
ax.set_title('Code Volume vs Defect Density\n(bubble size proportional to deploy %)',
             fontsize=13, fontweight='bold')
for x, y, n in zip(locs, dklocs, names_v):
    ax.annotate(n, (x, y), textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
write_figure(fig, 'volume_vs_defects.jpg')


### §12e  compliance.jpg

In [ ]:
compl_fig  = {s: _get_compl_pcts(DATA)[s] for s in MODEL_ORDER}
ranked_cf  = sorted(MODEL_ORDER, key=lambda s: compl_fig[s], reverse=True)
names_cf   = [SHORT_NAMES[s] for s in ranked_cf]
vals_cf    = [compl_fig[s] for s in ranked_cf]
colors_cf  = ['#2ecc71' if v>=80 else '#e67e22' if v>=65 else '#e74c3c' for v in vals_cf]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(names_cf, vals_cf, color=colors_cf)
ax.set_ylabel('Compliance (%)', fontsize=11)
ax.set_title('Requirements Compliance by Model (requirements-scanner)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 105)
ax.axhline(80, color='green',  linestyle='--', alpha=0.5, label='80% threshold')
ax.axhline(65, color='orange', linestyle='--', alpha=0.5, label='65% threshold')
for bar, v in zip(bars, vals_cf):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_xticklabels(names_cf, rotation=35, ha='right')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
write_figure(fig, 'compliance.jpg')
print(f"  --> All 5 matplotlib figures generated")


## §13  Summary

In [ ]:
print("=" * 70)
print("CHAPTER 7 GENERATION COMPLETE")
print("=" * 70)

tables   = [(t, p) for t, p in WRITTEN_FILES if t == 'table']
diagrams = [(t, p) for t, p in WRITTEN_FILES if t == 'diagram']
figures  = [(t, p) for t, p in WRITTEN_FILES if t == 'figure']

print(f"\n[TABLES] LaTeX table files ({len(tables)}):")
for _, p in tables:
    print(f"   {p.name}")

print(f"\n[DIAGRAMS] TikZ diagram files ({len(diagrams)}):")
for _, p in diagrams:
    print(f"   {p.name}")

print(f"\n[FIGURES] Matplotlib figures ({len(figures)}):")
for _, p in figures:
    print(f"   {p.name}")

print(f"\nTotal: {len(WRITTEN_FILES)} files generated from {DATA_JSON.resolve()}")
print("=" * 70)
